<a href="https://colab.research.google.com/github/singhakshit1402-lgtm/PATIENT_RECOVERY_ANALYSIS/blob/main/patient_recovery_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd


def load_data(filepath="healthcare_patient_analytics_seaborn.csv"):
    df = pd.read_csv(filepath)
    df["visit_date"] = pd.to_datetime(df["visit_date"])
    df["month"] = df["visit_date"].dt.to_period("M").astype(str)

    # Categorize recovery scores and readmission risk
    df["recovery_tier"] = pd.cut(
        df["recovery_score"],
        bins=[0, 60, 75, 85, 101],
        labels=["Low (<60)", "Moderate (60-74)", "Good (75-84)", "Excellent (85-100)"],
        right=False,
    )
    df["risk_level"] = pd.cut(
        df["readmission_risk"],
        bins=[0, 0.25, 0.50, 1.01],
        labels=["Low (<25%)", "Medium (25-49%)", "High (>=50%)"],
        right=False,
    )
    df["critical_watchlist"] = (df["recovery_score"] < 60) & (
        df["readmission_risk"] >= 0.50
    )
    return df


def executive_story(df):
    """Automatically computes and narrates what the dataset is telling."""
    total = len(df)
    avg_rec = df["recovery_score"].mean()
    avg_risk = df["readmission_risk"].mean() * 100
    avg_stay = df["length_of_stay_days"].mean()
    avg_cost = df["treatment_cost"].mean()

    dept_rec = df.groupby("department")["recovery_score"].mean()
    best_dept, best_val = dept_rec.idxmax(), dept_rec.max()
    worst_dept, worst_val = dept_rec.idxmin(), dept_rec.min()

    busiest_dept = df["department"].value_counts().idxmax()
    busiest_cnt = df["department"].value_counts().max()

    top_age = df["age_group"].value_counts().idxmax()
    top_age_pct = df["age_group"].value_counts(normalize=True).max() * 100

    routine_pct = (df["visit_type"] == "Routine").mean() * 100
    emerg_pct = (df["visit_type"] == "Emergency").mean() * 100

    corr_cost_rec = df["treatment_cost"].corr(df["recovery_score"])
    corr_stay_rec = df["length_of_stay_days"].corr(df["recovery_score"])

    low_rec_cnt = (df["recovery_score"] < 60).sum()
    high_risk_cnt = (df["readmission_risk"] >= 0.50).sum()
    crit_cnt = df["critical_watchlist"].sum()

    print("\n" + "=" * 65)
    print("        PATIENT RECOVERY ANALYSIS — AUTOMATED REPORT")
    print("=" * 65)
    print(f"Dataset Period   : {df['visit_date'].min().date()} to {df['visit_date'].max().date()}")
    print(f"Total Visits     : {total:,} patients (Missing values: {df.isnull().sum().sum()})")
    print(f"Avg Recovery     : {avg_rec:.2f} / 100")
    print(f"Avg Risk         : {avg_risk:.1f}% readmission probability")
    print(f"Avg Stay & Cost  : {avg_stay:.2f} days | Avg Cost: {avg_cost:,.2f}")
    print("-" * 65)
    print("WHAT THE DATA TELLS US:")
    print("1. Overall Recovery Health:")
    print(f"   - {(df['recovery_score'] >= 75).mean()*100:.1f}% achieved Good/Excellent recovery (75+).")
    print(f"   - {low_rec_cnt} patients ({low_rec_cnt/total*100:.1f}%) had Low recovery (<60).")
    print("2. Departmental Consistency:")
    print(f"   - Best avg recovery  : {best_dept} ({best_val:.2f})")
    print(f"   - Lowest avg recovery: {worst_dept} ({worst_val:.2f})")
    print(f"   - Gap is only {best_val - worst_val:.2f} points -> care quality is uniform across departments.")
    print(f"   - Busiest department : {busiest_dept} ({busiest_cnt:,} visits).")
    print("3. Patient Load & Demographics:")
    print(f"   - {routine_pct:.1f}% Routine visits vs {emerg_pct:.1f}% Emergency visits.")
    print(f"   - Largest age group is {top_age} ({top_age_pct:.1f}% of all visits).")
    print("4. Cost & Stay vs. Recovery:")
    print(f"   - Cost vs Recovery correlation : {corr_cost_rec:+.3f}")
    print(f"   - Stay vs Recovery correlation : {corr_stay_rec:+.3f}")
    print("   -> Higher cost or longer stay does NOT automatically mean better recovery.")
    print("5. Priority Action — Critical Watchlist:")
    print(f"   - {high_risk_cnt} patients ({high_risk_cnt/total*100:.1f}%) have High readmission risk (>=50%).")
    print(f"   - {crit_cnt} CRITICAL patients ({crit_cnt/total*100:.2f}%) have BOTH Low recovery (<60) & High risk (>=50%).")
    print("=" * 65)


def department_treatment_report(df):
    print("\n--- DEPARTMENT PERFORMANCE ---")
    dept = (
        df.groupby("department")
        .agg(
            Visits=("patient_id", "count"),
            Avg_Recovery=("recovery_score", "mean"),
            Avg_Risk_Pct=("readmission_risk", lambda x: x.mean() * 100),
            Avg_Stay=("length_of_stay_days", "mean"),
            Avg_Cost=("treatment_cost", "mean"),
            Critical_Cases=("critical_watchlist", "sum"),
        )
        .round(2)
        .sort_values("Avg_Recovery", ascending=False)
    )
    print(dept.to_string())

    print("\n--- TREATMENT TYPE PERFORMANCE ---")
    treat = (
        df.groupby("treatment_type")
        .agg(
            Visits=("patient_id", "count"),
            Avg_Recovery=("recovery_score", "mean"),
            Avg_Stay=("length_of_stay_days", "mean"),
            Avg_Cost=("treatment_cost", "mean"),
        )
        .round(2)
        .sort_values("Avg_Recovery", ascending=False)
    )
    print(treat.to_string())


def demographics_report(df):
    print("\n--- AGE GROUP BREAKDOWN ---")
    age = (
        df.groupby("age_group")
        .agg(
            Visits=("patient_id", "count"),
            Share_Pct=("patient_id", lambda x: len(x) / len(df) * 100),
            Avg_Recovery=("recovery_score", "mean"),
            Avg_Risk_Pct=("readmission_risk", lambda x: x.mean() * 100),
        )
        .round(2)
    )
    print(age.to_string())

    print("\n--- VISIT TYPE COMPARISON ---")
    vt = (
        df.groupby("visit_type")
        .agg(
            Visits=("patient_id", "count"),
            Share_Pct=("patient_id", lambda x: len(x) / len(df) * 100),
            Avg_Recovery=("recovery_score", "mean"),
            Avg_Stay=("length_of_stay_days", "mean"),
            Avg_Cost=("treatment_cost", "mean"),
        )
        .round(2)
    )
    print(vt.to_string())


def critical_watchlist_report(df):
    crit = df[df["critical_watchlist"]].sort_values(
        ["readmission_risk", "recovery_score"], ascending=[False, True]
    )
    cols = [
        "patient_id",
        "department",
        "age_group",
        "visit_type",
        "recovery_score",
        "readmission_risk",
        "treatment_cost",
    ]
    print(f"\n--- CRITICAL WATCHLIST: {len(crit)} PATIENTS (Recovery < 60 & Risk >= 50%) ---")
    print("Top 10 Highest-Priority Patients:")
    print(crit[cols].head(10).to_string(index=False))

    save = input("\nExport all 51 critical patients to CSV? (y/n): ").strip().lower()
    if save == "y":
        crit.to_csv("critical_watchlist_patients.csv", index=False)
        print("Saved to 'critical_watchlist_patients.csv'.")


def lookup_patient(df):
    try:
        pid = int(input("\nEnter Patient ID (1 to 5000): ").strip())
    except ValueError:
        print("Please enter a valid number.")
        return

    row = df[df["patient_id"] == pid]
    if row.empty:
        print(f"Patient ID {pid} not found.")
        return

    p = row.iloc[0]
    if p["critical_watchlist"]:
        verdict = "CRITICAL — Immediate post-discharge follow-up required"
    elif p["readmission_risk"] >= 0.50:
        verdict = "HIGH RISK — Schedule 7-day checkup"
    elif p["recovery_score"] < 60:
        verdict = "SLOW RECOVERY — Review treatment plan"
    else:
        verdict = "STABLE — Standard discharge protocol"

    print(f"\n--- Patient #{p['patient_id']} Diagnostic Card ---")
    print(f"Date / Visit     : {p['visit_date']} ({p['visit_type']})")
    print(f"Demographics     : {p['gender']}, Age {p['age_group']}, {p['region']} Region")
    print(f"Care Details     : {p['department']} | {p['treatment_type']} | {p['length_of_stay_days']} days")
    print(f"Treatment Cost   : {p['treatment_cost']:,.0f}")
    print(f"Recovery Score   : {p['recovery_score']:.0f}/100 ({p['recovery_tier']})")
    print(f"Readmission Risk : {p['readmission_risk']*100:.0f}% ({p['risk_level']})")
    print(f"System Verdict   : {verdict}")


def filter_by_department(df):
    depts = sorted(df["department"].unique())
    print("\nAvailable Departments:", ", ".join(depts))
    chosen = input("Enter department name: ").strip()
    matched = df[df["department"].str.lower() == chosen.lower()]

    if matched.empty:
        print("Department not found. Check spelling and try again.")
        return

    dept_name = matched["department"].iloc[0]
    print(f"\n--- {dept_name.upper()} QUICK SUMMARY ---")
    print(f"Total Visits       : {len(matched):,}")
    print(f"Avg Recovery Score : {matched['recovery_score'].mean():.2f} / 100")
    print(f"Avg Readmit Risk   : {matched['readmission_risk'].mean()*100:.1f}%")
    print(f"Avg Length of Stay : {matched['length_of_stay_days'].mean():.2f} days")
    print(f"Avg Treatment Cost : {matched['treatment_cost'].mean():,.2f}")
    print(f"Critical Patients  : {matched['critical_watchlist'].sum()}")
    print(f"Most Common Treat. : {matched['treatment_type'].value_counts().idxmax()}")


def main():
    df = load_data()
    while True:
        print("\n=========== HOSPITAL RECOVERY ANALYZER ===========")
        print("1. Automated Executive Report (What the data tells)")
        print("2. Department & Treatment Performance Tables")
        print("3. Age Group & Visit Type Breakdown")
        print("4. Critical Watchlist (Top High-Risk Patients)")
        print("5. Patient Diagnostic Lookup (by Patient ID)")
        print("6. Query a Specific Department")
        print("0. Exit")
        choice = input("Choose an option (0-6): ").strip()

        if choice == "1":
            executive_story(df)
        elif choice == "2":
            department_treatment_report(df)
        elif choice == "3":
            demographics_report(df)
        elif choice == "4":
            critical_watchlist_report(df)
        elif choice == "5":
            lookup_patient(df)
        elif choice == "6":
            filter_by_department(df)
        elif choice == "0":
            print("Exiting analyzer.")
            break
        else:
            print("Invalid choice. Enter a number from 0 to 6.")


if __name__ == "__main__":
    main()


=========== HOSPITAL RECOVERY ANALYZER ===========
1. Automated Executive Report (What the data tells)
2. Department & Treatment Performance Tables
3. Age Group & Visit Type Breakdown
4. Critical Watchlist (Top High-Risk Patients)
5. Patient Diagnostic Lookup (by Patient ID)
6. Query a Specific Department
0. Exit
Choose an option (0-6): 1

        PATIENT RECOVERY ANALYSIS — AUTOMATED REPORT
Dataset Period   : 2022-01-01 to 2022-07-28
Total Visits     : 5,000 patients (Missing values: 0)
Avg Recovery     : 74.72 / 100
Avg Risk         : 28.1% readmission probability
Avg Stay & Cost  : 4.06 days | Avg Cost: 54,915.47
-----------------------------------------------------------------
WHAT THE DATA TELLS US:
1. Overall Recovery Health:
   - 50.9% achieved Good/Excellent recovery (75+).
   - 516 patients (10.3%) had Low recovery (<60).
2. Departmental Consistency:
   - Best avg recovery  : Neurology (75.18)
   - Lowest avg recovery: Cardiology (74.15)
   - Gap is only 1.03 points -> care q